# Data Mining Project — Political Side & Party from Sejm Speeches

**Goal:** from an MP speech transcript, predict:
1. political **side** (`left` / `center` / `right`)
2. parliamentary **club / party**

**Full dataset used in this notebook:** Sejm terms **9 + 10 combined**  
(`data/combined/` — all labeled speeches from both terms)

**Final model:** word+char **TF-IDF** + balanced Logistic Regression  
**Split:** speaker-wise (`term_memberId`) on the **full combined** data

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "pyproject.toml").exists():
            ROOT = parent
            break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA9 = ROOT / "data" / "term9"
DATA10 = ROOT / "data" / "term10"
COMBINED = ROOT / "data" / "combined"
SPLITS = COMBINED / "splits"
MODELS = ROOT / "models"
SIDE_ORDER = ["left", "center", "right"]

plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.25})

ready = {
    "term9_labeled": (DATA9 / "dataset_labeled.csv").exists(),
    "term10_labeled": (DATA10 / "dataset_labeled.csv").exists(),
    "combined": (COMBINED / "dataset_labeled.csv").exists(),
    "splits": (SPLITS / "test.csv").exists(),
    "side_model": (MODELS / "side_tfidf.joblib").exists(),
    "club_model": (MODELS / "club_tfidf.joblib").exists(),
}
print("ROOT:", ROOT)
for key, ok in ready.items():
    print(f"  {key}: {'OK' if ok else 'MISSING'}")
if not ready["combined"]:
    print("\nFull dataset not ready. After term 10 download, run:")
    print("  make prepare-term SEJM_TERM=10")
    print("  make prepare-term SEJM_TERM=9")
    print("  make prepare-combined")
    print("  make train")

## 1. Pipeline (full dataset)

| Step | Description |
|------|-------------|
| Fetch | Speeches + MPs for **term 9** and **term 10** |
| Clean | Strip procedural boilerplate; drop oaths / very short texts |
| Label | Map club → side; keep club for party classification |
| **Combine** | **Merge both terms into one dataset** (`data/combined/`) |
| Split | 70/15/15 by speaker on the **combined** data |
| Train | TF-IDF (word 1–2 + char 3–5) + OvR Logistic Regression for **side** and **club** |
| Predict | CLI: `predict-side`, `predict-club` |

## 2. Full dataset overview (term 9 + term 10)

Primary table/plots below use **`data/combined/dataset_labeled.csv`** when available.

In [ ]:
def load_labeled(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return pd.read_csv(path)


df9 = load_labeled(DATA9 / "dataset_labeled.csv")
df10 = load_labeled(DATA10 / "dataset_labeled.csv")
df = load_labeled(COMBINED / "dataset_labeled.csv")

rows = []
for name, frame in (("term9 only", df9), ("term10 only", df10), ("FULL combined 9+10", df)):
    if frame is None:
        continue
    rows.append(
        {
            "dataset": name,
            "speeches": len(frame),
            "speakers": (
                frame["speaker_key"].nunique()
                if "speaker_key" in frame.columns
                else frame["member_id"].nunique()
            ),
            "clubs": frame["club"].nunique(),
            "left": int((frame["side"] == "left").sum()),
            "center": int((frame["side"] == "center").sum()),
            "right": int((frame["side"] == "right").sum()),
        }
    )

overview = pd.DataFrame(rows)
display(overview)

if df is None:
    raise SystemExit(
        "Full combined dataset missing. Prepare term 10, then: make prepare-combined"
    )

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

side_counts = df["side"].value_counts().reindex(SIDE_ORDER)
axes[0].bar(side_counts.index, side_counts.values, color=["#c44e52", "#4c72b0", "#55a868"])
axes[0].set_title("Side — FULL dataset")
axes[0].set_ylabel("speeches")

if "term" in df.columns:
    term_side = (
        df.groupby(["term", "side"])
        .size()
        .unstack("side", fill_value=0)
        .reindex(columns=SIDE_ORDER, fill_value=0)
    )
    term_side.plot(kind="bar", ax=axes[1], color=["#c44e52", "#4c72b0", "#55a868"], rot=0)
    axes[1].set_title("Side by term")
    axes[1].set_xlabel("term")

club_counts = df["club"].value_counts().head(10)
axes[2].barh(club_counts.index[::-1], club_counts.values[::-1])
axes[2].set_title("Top clubs — FULL dataset")
fig.tight_layout()
plt.show()

print(f"FULL dataset speeches: {len(df):,}")
print(f"FULL dataset clubs: {', '.join(sorted(df['club'].unique()))}")

## 3. Earlier experiments (term 9 only) — why we kept TF-IDF

These runs were done **before** combining term 10. Same term-9 splits → comparable ablations.

| Model | Accuracy | Macro F1 | Left F1 | Left recall | Notes |
|-------|---------:|---------:|--------:|------------:|-------|
| **TF-IDF raw** | **0.726** | **0.670** | **0.436** | 0.374 | Best overall → chosen for full dataset |
| TF-IDF + left bias (0.41) | 0.578 | 0.582 | 0.441 | **0.815** | Recall↑, overall↓ — not used in final CLI |
| HerBERT frozen + LR | 0.603 | 0.570 | 0.379 | 0.426 | Weaker than TF-IDF |
| HerBERT + left bias | 0.564 | 0.554 | 0.408 | 0.623 | Still below TF-IDF |

**Decision for the full 9+10 project:** train/report **raw TF-IDF** on the combined dataset; add term 10 to help the weak `left` class.

In [ ]:
comparison = pd.DataFrame(
    [
        {"model": "TF-IDF raw", "accuracy": 0.726, "macro_f1": 0.670, "left_f1": 0.436, "left_recall": 0.374},
        {"model": "TF-IDF + left bias", "accuracy": 0.578, "macro_f1": 0.582, "left_f1": 0.441, "left_recall": 0.815},
        {"model": "HerBERT raw", "accuracy": 0.603, "macro_f1": 0.570, "left_f1": 0.379, "left_recall": 0.426},
        {"model": "HerBERT + left bias", "accuracy": 0.564, "macro_f1": 0.554, "left_f1": 0.408, "left_recall": 0.623},
    ]
)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comparison))
width = 0.2
for i, col in enumerate(["accuracy", "macro_f1", "left_f1", "left_recall"]):
    ax.bar(x + (i - 1.5) * width, comparison[col], width, label=col)
ax.set_xticks(x)
ax.set_xticklabels(comparison["model"], rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Ablation on term 9 only (historical)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()
display(comparison)

## 4. Final side results — FULL combined dataset (9+10)

Evaluated on `data/combined/splits/test.csv` with `models/side_tfidf.joblib`.

In [ ]:
metrics_path = MODELS / "baseline_metrics.json"
side_model_path = MODELS / "side_tfidf.joblib"
club_model_path = MODELS / "club_tfidf.joblib"

if not (SPLITS / "test.csv").exists():
    raise SystemExit("Combined splits missing. Run: make prepare-combined")
if not side_model_path.exists():
    raise SystemExit("Side model missing. Run: make train")

test = pd.read_csv(SPLITS / "test.csv")
print(
    f"Test split (FULL dataset): {len(test):,} speeches | "
    f"speakers={test.get('speaker_key', test['member_id']).nunique()} | "
    f"terms={sorted(test['term'].unique().tolist()) if 'term' in test.columns else '?'}"
)

side_clf = joblib.load(side_model_path)
y_true_side = test["side"]
y_pred_side = side_clf.predict(test["text"])

acc = float((y_pred_side == y_true_side).mean())
macro = float(f1_score(y_true_side, y_pred_side, average="macro", zero_division=0))
print(f"=== SIDE on FULL test === accuracy={acc:.3f} macro_f1={macro:.3f}")
print(classification_report(y_true_side, y_pred_side, digits=3, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_true_side, y_pred_side, labels=SIDE_ORDER, ax=axes[0], colorbar=False
)
axes[0].set_title("Side CM — FULL dataset test")

report = classification_report(
    y_true_side, y_pred_side, digits=3, zero_division=0, output_dict=True
)
f1s = [report[label]["f1-score"] for label in SIDE_ORDER]
axes[1].bar(SIDE_ORDER, f1s, color=["#c44e52", "#4c72b0", "#55a868"])
axes[1].set_ylim(0, 1)
axes[1].set_title("Side F1 per class — FULL dataset")
for i, v in enumerate(f1s):
    axes[1].text(i, v + 0.02, f"{v:.2f}", ha="center")
fig.tight_layout()
plt.show()

## 5. Party / club recognition — FULL combined dataset

Same TF-IDF pipeline, target = `club`, trained on **all clubs present in terms 9+10**.

In [ ]:
if not club_model_path.exists():
    raise SystemExit("Club model missing. Run: make train")

club_clf = joblib.load(club_model_path)
y_true_club = test["club"]
y_pred_club = club_clf.predict(test["text"])

acc_c = float((y_pred_club == y_true_club).mean())
macro_c = float(f1_score(y_true_club, y_pred_club, average="macro", zero_division=0))
print(f"=== CLUB on FULL test === accuracy={acc_c:.3f} macro_f1={macro_c:.3f}")
print(classification_report(y_true_club, y_pred_club, digits=3, zero_division=0))

top_clubs = y_true_club.value_counts().head(8).index.tolist()
mask = y_true_club.isin(top_clubs).to_numpy()
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true_club.to_numpy()[mask],
    np.asarray(y_pred_club)[mask],
    labels=top_clubs,
    ax=ax,
    colorbar=False,
    xticks_rotation=45,
)
ax.set_title("Club CM (top-8) — FULL dataset test")
fig.tight_layout()
plt.show()

## 6. Live demo (models trained on FULL dataset)

In [ ]:
from data_mining_project.utils.sejm_text import clean_speech_text

demos = [
    "Musimy wzmocnić państwo opiekuńcze, podnieść płacę minimalną i chronić prawa pracownicze.",
    "Potrzebujemy kompromisu w sprawie praworządności i współpracy z Unią Europejską.",
    "Należy obniżyć podatki, ograniczyć biurokrację i bronić tradycyjnych wartości.",
]

for text in demos:
    cleaned = clean_speech_text(text) or text
    side = side_clf.predict([cleaned])[0]
    side_probs = dict(zip(side_clf.classes_, side_clf.predict_proba([cleaned])[0]))
    club = club_clf.predict([cleaned])[0]
    club_probs = sorted(
        zip(club_clf.classes_, club_clf.predict_proba([cleaned])[0]),
        key=lambda x: -x[1],
    )[:3]
    print(f"[{side} | {club}] {text}")
    print("  side:", ", ".join(f"{k}={v:.2f}" for k, v in sorted(side_probs.items(), key=lambda x: -x[1])))
    print("  club top3:", ", ".join(f"{k}={v:.2f}" for k, v in club_probs))
    print()

## 7. Conclusions

1. The **reported system** is trained and evaluated on the **full combined dataset (terms 9 + 10)**.
2. Term-9 ablations showed **TF-IDF > HerBERT** and that aggressive left-bias calibration hurts overall quality.
3. Adding term 10 increases speaker diversity (especially for left) on top of the classical TF-IDF model.
4. **Club/party** prediction is the harder multi-class task on the same full dataset.

### Reproduce full-dataset results
```bash
make prepare-term SEJM_TERM=10
make prepare-term SEJM_TERM=9
make prepare-combined
make train
```
See also [`COMMANDS.md`](../../../COMMANDS.md).